# AryColBring Training Pipeline — Annotated Walkthrough

**Source script:** [`t01_advisor.py`](https://github.com/masterofray/cooprecsys/blob/dev/test/arycolbring_tests/t01_advisor.py) from the `cooprecsys` repository

This notebook takes the original `AryColBring_Train_Test` **class** from `t01_advisor.py` and rewrites its logic as a series of small, independent, well-commented **functions** — one function per pipeline step — so someone new to the codebase can read it top-to-bottom without tracing method calls back and forth inside a class.

**What this pipeline does, in one sentence:** it loads employee/product interaction data, converts it into an implicit "rating" signal, splits it into train/test sets, trains a hybrid collaborative-filtering recommendation model (`AryColBring`), and saves the trained model plus a training report to disk.

### How this notebook is organized
1. Background — what problem is this code solving?
2. Imports — what each library/module is for
3. Step-by-step **functions** that mirror every method of the original class (real logic, reorganized)
4. A **runnable mini-demo** using synthetic data and lightweight stand-ins, so you can execute the whole pipeline end-to-end even without the real `cooprecsys` package installed
5. How the original class glues everything together (`__call__` → `run_full_pipeline()`)
6. The command-line interface (`main()`)
7. Function-based vs. class-based design — why the original author likely chose a class
8. Appendix — the original class, unmodified, for side-by-side comparison

> ⚠️ **Note on runnability:** The real modules imported at the top of the original script (`configs`, `db`, `features`, `prepare`, `qrates`, `models.arycolbring`) live inside the private `cooprecsys` repository and are **not available in this environment**. Section 3 shows the *real* logic, faithfully reorganized into functions, for reading and reference — those specific cells are marked "reference only" and are not executed. Section 4 provides small stand-ins for the heavy dependencies so you can actually press "Run All" and watch the pipeline execute end-to-end on toy data.


## 1. Background: what is this pipeline for?

`AryColBring` is a **hybrid collaborative filtering** model. "Hybrid" means it doesn't only look at *who interacted with what* (pure collaborative filtering) — it also uses **side features** about the users and the items, similar in spirit to the LightFM approach:

- **User features**, e.g. `EmployeeAge`, `EmployeeGender`, `Resistant`, `IsAllergic`, `VitalityDays`
- **Item features**, e.g. `ProductPrice`, `Quantity`, `Discount`, `TotalPrice`, `Class`

Users and items get embedded into a shared latent space, and the side features let the model make reasonable predictions even for users/items with very few past interactions (the "cold start" problem).

The training pipeline in `t01_advisor.py` performs these stages, in order:

| Stage | Method in original class | What it does |
|---|---|---|
| 1 | `Data` (property setter) | Validate & load the raw interaction data (CSV/Parquet/etc.) |
| 2 | `_RateProgress` | Turn raw interactions into a "quasi-rating" (an implicit rating score) |
| 3 | `_RatePosthoc` | Auto-detect user/item ID columns, merge the rating back with the raw data |
| 4 | `_brokedata` | Randomly split into train/test sets |
| 5 | `_ttprocess` (called from inside `_brokedata`) | Convert each split into sparse matrices: `(interactions, user_features, item_features, weights)` |
| 6 | `train` | Fit the `AryColBring` model and generate a training report |
| 7 | `save` | Persist the trained model to disk |
| 8 | `__call__` | Run stages 2–7 in sequence and return a summary dictionary |

We'll turn each stage into its own function below, in the same order.


## 2. Imports

First, the standard-library and third-party imports — these run in any normal Python environment. Then the project-internal imports, which only exist inside the private `cooprecsys` repo, shown as reference.


In [ ]:
# --- Standard library & third-party imports (these run in any normal Python environment) ---
import gc                       # manual garbage collection, used after big DataFrame operations
import time                     # for timing how long training takes
import numpy  as np             # numerical arrays, random splitting
import pandas as pd             # tabular data (the interactions dataset)
from pathlib  import Path       # filesystem paths, cross-platform
from copy     import deepcopy   # safe copies of DataFrames / paths so mutation doesn't leak
from datetime import datetime   # used to timestamp saved model files
from typing   import Optional, Tuple, Union, List, Dict  # type hints for readability

print("Standard imports OK")


```python
# --- Project-internal imports (reference only -- these live inside the private cooprecsys repo) ---
import sys
LocDir = Path(__file__).resolve().parents[2] / 'src'
sys.path.append(str(LocDir))

from configs  import _cfg, logger
from db       import duckdb_connection
from features import load_data
from prepare  import DetectReco_Identifier
from qrates   import GenQuasi_Lazy, DMD
from models.arycolbring import AryColBringModelTrainer as ACBmodel
from models.arycolbring.assist import fileload_interactions, describe_interactions
```

**What each project-internal import is for:**

| Import | Role |
|---|---|
| `_cfg`, `logger` | Global config parser (reads an `.ini`/`.cfg` file) and a shared logger instance |
| `duckdb_connection` | A DuckDB connection helper, used elsewhere in the project for fast SQL-style aggregation |
| `load_data` | Reads a CSV/Parquet file into a validated `pandas.DataFrame` |
| `DetectReco_Identifier` | Inspects column names and guesses which ones are the user-ID, item-ID, quantity, total-price, and discount columns |
| `GenQuasi_Lazy` | Computes an implicit "quasi-rating" score from raw interaction rows (e.g. combining quantity, recency, price into one score) |
| `DMD` | Turns a DataFrame + feature-column lists into the sparse matrices a LightFM-style model expects: `(interactions, user_features, item_features, weights)` |
| `ACBmodel` (`AryColBringModelTrainer`) | The actual trainable recommendation model class |
| `fileload_interactions`, `describe_interactions` | Small helpers to load/describe interaction files (imported but unused directly in this script) |

**If you want to run this against the real package**, install it first:
```bash
pip install git+https://github.com/masterofray/cooprecsys.git@dev
```
then in Section 3 swap the mock stand-ins for the real imports above, point `DATA_PATH` at your real Parquet/CSV file, and skip Section 4's synthetic demo.


## 3. The pipeline, one function per step

Below, every method of `AryColBring_Train_Test` becomes a standalone function. This is the **same logic**, just:
- no `self.` — everything is passed in as arguments and returned as outputs
- one clear docstring per function explaining *what* and *why*
- inline comments on the trickier lines

These cells import the private project modules, so they are written as **reference/annotated code** (not executed directly here). Section 4 gives you a runnable version with mock stand-ins.


### 3.1 Configuration loading

Original method: `_load_config`. Reads model hyperparameters from a config file, falling back to sensible defaults if a key is missing.


```python
def load_config(_cfg) -> dict:
    """
    Read model hyperparameters from the project's global config object.

    Parameters
    ----------
    _cfg : configparser.ConfigParser
        The shared config object loaded from an .ini file.

    Returns
    -------
    dict
        Hyperparameters ready to be unpacked into the model constructor.
    """
    verbose = _cfg.get('logging', 'level') in ['DEBUG', 'INFO']
    config = {
        "no_components"    : _cfg.getint('model', "no_components", fallback=10),
        "loss"             : _cfg.get('model', "loss", fallback="warp"),
        "learning_rate"    : _cfg.getfloat('model', "learning_rate", fallback=0.05),
        "epochs"           : _cfg.getint('model', "epochs", fallback=10),
        "num_threads"      : _cfg.getint('model', "num_threads", fallback=4),
        "dtype"            : _cfg.get('model', "dtype", fallback="float32"),
        "learning_schedule": _cfg.get('model', "learning_schedule", fallback="adagrad"),
        "verbosity"        : verbose,
    }
    return config
```

**Beginner note:** `fallback=...` means "if this key is missing from the config file, use this default instead of crashing." This is a defensive pattern worth copying in your own config-loading code.


### 3.2 Data loading & validation

Original method: the `Data` property setter. In the class this is triggered automatically when you write `instance.Data = "path/to/file.csv"`. As a function it becomes explicit and easier to unit-test.


```python
def load_and_validate_data(value, load_data_fn, logger) -> pd.DataFrame:
    """
    Accepts either a file path or an already-loaded DataFrame, validates it,
    and returns a clean DataFrame ready for the pipeline.

    Validation rules:
      - a path must exist, be a file (not a folder), and be non-empty
      - a file must have at least 20 lines (a crude 'enough data to bother' check)
      - a DataFrame must be non-empty and have at least 20 rows
    """
    if not isinstance(value, (str, Path, pd.DataFrame)):
        raise TypeError(
            f"Invalid data type: {type(value).__name__}. "
            "Expected str, Path, or pandas.DataFrame."
        )

    if isinstance(value, (str, Path)):
        path_obj = Path(value)
        if not path_obj.exists():
            raise FileNotFoundError(f"File does not exist at '{path_obj}'.")
        if not path_obj.is_file():
            raise ValueError(f"Target path is a directory, not a file -> '{path_obj}'.")
        if path_obj.stat().st_size == 0:
            raise ValueError(f"The file '{path_obj.name}' is empty.")

        with path_obj.open("r", encoding="utf-8", errors="ignore") as f:
            line_count = sum(1 for _ in f)
        if line_count < 20:
            raise ValueError(
                f"File has only {line_count} lines. A minimum of 20 lines is required."
            )

        data = load_data_fn(data_path=path_obj, memory_limit="16GB")
        logger.debug(f"Successfully loaded data from '{path_obj}'.")
        return data

    # value is already a DataFrame
    if value.empty:
        raise ValueError("The provided DataFrame is empty.")
    if len(value) < 20:
        raise ValueError(f"DataFrame has only {len(value)} rows; minimum is 20.")
    logger.debug("Successfully validated an in-memory DataFrame.")
    return value
```

**Beginner note:** the original code wraps the file-reading branch in a `try/except/finally` block, but the `finally` clause always runs `load_data(...)` regardless of whether an exception was raised — that's very likely an unintentional bug in the original (a `finally` that overrides the point of the `try`). The function above fixes that by only loading after all checks pass, which is the behavior actually intended.


### 3.3 Default feature lists

Small helper for the constructor defaults (`UserFeats` / `ItemFeats` when `None` is passed in).


```python
def default_user_features() -> list:
    return ['EmployeeAge', 'EmployeeGender', 'Resistant', 'IsAllergic', 'VitalityDays']

def default_item_features() -> list:
    return ['ProductPrice', 'Quantity', 'Discount', 'TotalPrice', 'Class']
```


### 3.4 Quasi-rating generation

Original method: `_RateProgress`. Converts raw interaction rows into an implicit rating score via `GenQuasi_Lazy`.


```python
def generate_quasi_ratings(data: pd.DataFrame, GenQuasi_Lazy, logger) -> pd.DataFrame:
    """
    Turn raw interaction rows (e.g. one row per purchase/click) into an
    implicit 'quasi-rating' score the model can be trained on, since this
    dataset has no explicit 1-5 star ratings.
    """
    data_rate = GenQuasi_Lazy(data)
    logger.info("Data loaded: shape = %s", data_rate.shape)
    logger.debug("Columns: %s", data_rate.columns.tolist())
    return data_rate
```


### 3.5 Column detection & merge

Original method: `_RatePosthoc`. Auto-detects which columns are the user ID / item ID / quantity / total / discount columns, merges the computed rating back onto the raw data, and extends the item-feature list with the newly detected numeric columns.


```python
def detect_and_merge(data: pd.DataFrame,
                      data_rate: pd.DataFrame,
                      item_feats: list,
                      DetectReco_Identifier) -> tuple:
    """
    Detect key columns (user/item id, quantity, total price, discount) and
    merge the quasi-rating DataFrame back onto the original data.

    Returns
    -------
    (collect, data_merge, item_feats) :
        collect     -> dict of detected column names, e.g. {'user_col': 'EmployeeID', ...}
        data_merge  -> data with the quasi-rating column joined in
        item_feats  -> the item feature list, extended with detected numeric columns
    """
    collect = DetectReco_Identifier(data.columns.to_numpy())

    data_merge = data_rate.merge(
        data, on=[collect['user_col'], collect['item_col']]
    )

    extra_cols = [collect["quantity_col"], collect["total_col"], collect["discount_col"]]
    item_feats = list(set(item_feats + extra_cols))
    item_feats = [c for c in item_feats if c is not None]  # drop any that weren't detected

    return collect, data_merge, item_feats
```


### 3.6 Train / test split

Original methods: `_brokedata` and `_ttprocess`. Splits the merged data with a random boolean mask, then converts each split into the sparse-matrix format the model needs.


```python
def to_model_matrices(data: pd.DataFrame, collect: dict, user_feats: list,
                       item_feats: list, DMD) -> tuple:
    """
    Convert a DataFrame split into the 4 sparse matrices a LightFM-style
    model expects: (interactions, user_features, item_features, weights).
    """
    return DMD(
        data              = data,
        user_col          = collect['user_col'],
        item_col          = collect['item_col'],
        user_feature_cols = user_feats,
        item_feature_cols = item_feats,
    )


def split_train_test(data_merge: pd.DataFrame, test_ratio: float, collect: dict,
                      user_feats: list, item_feats: list, DMD) -> tuple:
    """
    Randomly split data_merge into train/test using a boolean mask (not
    scikit-learn's train_test_split -- same idea, hand-rolled with a fixed
    random seed for reproducibility), then convert both splits to matrices.
    """
    np.random.seed(4)  # fixed seed => same split every run, reproducible experiments
    n_rows = data_merge.shape[0]
    mask = np.random.rand(n_rows) > test_ratio   # True -> goes to train

    train_data = deepcopy(data_merge[mask])
    test_data  = deepcopy(data_merge[~mask])

    train_matrices = to_model_matrices(train_data, collect, user_feats, item_feats, DMD)
    test_matrices  = to_model_matrices(test_data,  collect, user_feats, item_feats, DMD)

    del mask, train_data, test_data
    gc.collect()  # free memory immediately rather than waiting for Python's GC cycle

    return train_matrices, test_matrices
```

**Beginner note:** `np.random.rand(n_rows) > test_ratio` creates one random number per row; if that number is bigger than `test_ratio` (e.g. 0.2) the row goes to train, otherwise to test. With `test_ratio = 0.2` roughly 80% of rows end up `True` (train) and 20% `False` (test) — but not *exactly* 80/20, since it's random per-row rather than an exact split.


### 3.7 Training

Original method: `train`. Instantiates the model with hyperparameters from config, fits it on the training matrices while validating against the test matrices each epoch, then writes a training report.


```python
def train_model(train_matrices: tuple, test_matrices: tuple, config: dict,
                 epochs: int, exname: str, output_dir: Path, ACBmodel, logger):
    """
    Fit the AryColBring model.

    train_matrices / test_matrices are each a 4-tuple:
        (interactions, user_features, item_features, sample_weight)
    """
    logger.info("Starting training: epochs=%d | threads=%d", epochs, config["num_threads"])

    model = ACBmodel(
        no_components     = config["no_components"],
        loss              = config["loss"],
        learning_rate     = config["learning_rate"],
        item_alpha        = 0.01,   # L2 regularization on item embeddings
        user_alpha        = 0.01,   # L2 regularization on user embeddings
        learning_schedule = config["learning_schedule"],
        random_state      = 4,
    )

    model.fit(
        interactions    = train_matrices[0],
        user_features   = train_matrices[1],
        item_features   = train_matrices[2],
        sample_weight   = train_matrices[3],
        epochs          = epochs,
        num_threads     = config["num_threads"],
        verbose         = config["verbosity"],
        validation_data = test_matrices[0],
        evaluate_every  = 1,   # compute a validation metric every epoch
    )

    report_path = model.generate_training_report(
        output_dir      = str(output_dir),
        experiment_name = exname,
    )
    logger.debug("Training completed. Report: %s", report_path)
    return model, report_path
```


### 3.8 Saving the model

Original method: `save`. Timestamps and writes the trained model weights to disk as an `.npz` file.


```python
def save_model(model, output_dir: Path, logger) -> Path:
    """
    Persist the trained model under output_dir/ACBmodel/YYYYMMDD_models.npz
    """
    dates = f'{datetime.now():%Y%m%d}'
    model_path = output_dir / "ACBmodel" / f'{dates}_models.npz'
    model_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_path))
    logger.info("Model saved: %s", model_path)
    return model_path
```


## 4. Runnable mini-demo (synthetic data + mock dependencies)

Everything above is the *real* logic but can't execute here, since it needs the private `cooprecsys` package. This section builds tiny stand-ins for `GenQuasi_Lazy`, `DetectReco_Identifier`, `DMD`, and `ACBmodel` — simplified but behaviorally similar — purely so you can **run the pipeline end-to-end** on toy data and see how the pieces connect.

If you install the real `cooprecsys` package, you can delete this section and call the Section 3 functions directly with the real imports.


In [ ]:
import logging

# A minimal logger stand-in (the real project uses a configured logger from `configs`)
logger = logging.getLogger("acb_demo")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


### 4.1 Synthetic dataset

We simulate a small "employee buys product" interaction table — the same shape of data the real pipeline expects: one row per interaction, with a user ID, item ID, and some user/item features.


In [ ]:
np.random.seed(0)

N_USERS = 40
N_ITEMS = 15
N_ROWS  = 400

raw_data = pd.DataFrame({
    "EmployeeID"     : np.random.randint(1, N_USERS + 1, size=N_ROWS),
    "ProductID"      : np.random.randint(1, N_ITEMS + 1, size=N_ROWS),
    "EmployeeAge"    : np.random.randint(20, 55, size=N_ROWS),
    "EmployeeGender" : np.random.choice(["M", "F"], size=N_ROWS),
    "Resistant"      : np.random.choice([0, 1], size=N_ROWS),
    "IsAllergic"     : np.random.choice([0, 1], size=N_ROWS),
    "VitalityDays"   : np.random.randint(0, 30, size=N_ROWS),
    "ProductPrice"   : np.round(np.random.uniform(5, 100, size=N_ROWS), 2),
    "Quantity"       : np.random.randint(1, 10, size=N_ROWS),
    "Discount"       : np.round(np.random.uniform(0, 0.3, size=N_ROWS), 2),
    "Class"          : np.random.choice(["A", "B", "C"], size=N_ROWS),
})
raw_data["TotalPrice"] = (raw_data["ProductPrice"] * raw_data["Quantity"] *
                           (1 - raw_data["Discount"])).round(2)

print(f"Synthetic dataset: {raw_data.shape[0]} rows, {raw_data.shape[1]} columns")
raw_data.head()


### 4.2 Mock `GenQuasi_Lazy` — quasi-rating generation

The real function likely blends quantity, spend, and recency into one implicit score. Our stand-in does something simple but analogous: a rating that rewards higher quantity and total spend, scaled into a 1–5 range.


In [ ]:
def mock_gen_quasi_lazy(data: pd.DataFrame) -> pd.DataFrame:
    """
    Simplified stand-in for the real GenQuasi_Lazy().
    Produces one row per (EmployeeID, ProductID) pair with a 'quasi_rating' column.
    """
    grouped = (
        data.groupby(["EmployeeID", "ProductID"])
        .agg(total_spend=("TotalPrice", "sum"), total_qty=("Quantity", "sum"))
        .reset_index()
    )
    # Normalize spend into a 1-5 'implicit rating' scale
    max_spend = grouped["total_spend"].max()
    grouped["quasi_rating"] = 1 + 4 * (grouped["total_spend"] / max_spend)
    return grouped[["EmployeeID", "ProductID", "quasi_rating"]]


data_rate = mock_gen_quasi_lazy(raw_data)
print("Quasi-rating table:", data_rate.shape)
data_rate.head()


### 4.3 Mock `DetectReco_Identifier` — column auto-detection

The real version likely uses fuzzy name-matching to guess which column is the user id, item id, etc. Our stand-in does a simple keyword match, which is enough to demonstrate the idea.


In [ ]:
def mock_detect_reco_identifier(columns) -> dict:
    """
    Simplified stand-in for DetectReco_Identifier().
    Scans column names for keywords and returns the detected roles.
    """
    cols = list(columns)

    def find(keyword_options):
        for kw in keyword_options:
            for c in cols:
                if kw.lower() in c.lower():
                    return c
        return None

    return {
        "user_col"    : find(["employee", "user"]),
        "item_col"    : find(["product", "item"]),
        "quantity_col": find(["quantity"]),
        "total_col"   : find(["totalprice", "total"]),
        "discount_col": find(["discount"]),
    }


collect = mock_detect_reco_identifier(raw_data.columns)
collect


### 4.4 Merge + extend item features

This step is identical to the real logic in Section 3.5 — no mocking needed, since it's plain pandas.


In [ ]:
def detect_and_merge(data: pd.DataFrame, data_rate: pd.DataFrame,
                      item_feats: list, collect: dict) -> tuple:
    """Merge the quasi-rating table back onto the raw data and extend item_feats."""
    data_merge = data_rate.merge(data, on=[collect['user_col'], collect['item_col']])

    extra_cols = [collect["quantity_col"], collect["total_col"], collect["discount_col"]]
    item_feats = list(set(item_feats + extra_cols))
    item_feats = [c for c in item_feats if c is not None]

    return data_merge, item_feats


user_feats = ['EmployeeAge', 'EmployeeGender', 'Resistant', 'IsAllergic', 'VitalityDays']
item_feats = ['ProductPrice', 'Quantity', 'Discount', 'TotalPrice', 'Class']

data_merge, item_feats = detect_and_merge(raw_data, data_rate, item_feats, collect)
print("Merged shape:", data_merge.shape)
print("Extended item_feats:", item_feats)
data_merge.head()


### 4.5 Mock `DMD` — build model-ready matrices, then split train/test

Real `DMD` would return sparse `scipy` matrices. To keep the demo simple and inspectable, our mock version returns plain Python dictionaries carrying the same conceptual pieces: interactions, user features, item features, weights.


In [ ]:
def mock_dmd(data: pd.DataFrame, user_col: str, item_col: str,
             user_feature_cols: list, item_feature_cols: list) -> dict:
    """
    Simplified stand-in for DMD(). A real implementation would build sparse
    COO/CSR matrices; here we keep a small dict so it's easy to inspect.
    """
    return {
        "interactions": data[[user_col, item_col, "quasi_rating"]].reset_index(drop=True),
        "user_features": data[[user_col] + user_feature_cols].drop_duplicates(user_col),
        "item_features": data[[item_col] + item_feature_cols].drop_duplicates(item_col),
        "weight": data["quasi_rating"].values,
    }


def split_train_test_demo(data_merge: pd.DataFrame, test_ratio: float, collect: dict,
                           user_feats: list, item_feats: list) -> tuple:
    """Same random-mask split logic as the real _brokedata(), using mock_dmd()."""
    np.random.seed(4)
    n_rows = data_merge.shape[0]
    mask = np.random.rand(n_rows) > test_ratio

    train_df = deepcopy(data_merge[mask])
    test_df  = deepcopy(data_merge[~mask])

    train_matrices = mock_dmd(train_df, collect['user_col'], collect['item_col'], user_feats, item_feats)
    test_matrices  = mock_dmd(test_df,  collect['user_col'], collect['item_col'], user_feats, item_feats)

    del mask, train_df, test_df
    gc.collect()
    return train_matrices, test_matrices


TEST_RATIO = 0.2
train_matrices, test_matrices = split_train_test_demo(data_merge, TEST_RATIO, collect, user_feats, item_feats)

print(f"Train interactions: {len(train_matrices['interactions'])} rows")
print(f"Test  interactions: {len(test_matrices['interactions'])} rows")


### 4.6 Mock `ACBmodel` — training loop

A real matrix-factorization fit would run gradient updates over latent embeddings. Our mock model just tracks a fake decreasing "loss" per epoch and stores placeholder embeddings, so `.fit()` and `.save_model()` behave the same way from the *outside* — which is exactly what this notebook is trying to teach: the shape of the pipeline, not the internals of matrix factorization.


In [ ]:
class MockACBModel:
    """Stand-in for AryColBringModelTrainer -- mimics its public interface only."""

    def __init__(self, no_components, loss, learning_rate, item_alpha,
                 user_alpha, learning_schedule, random_state):
        self.no_components = no_components
        self.loss = loss
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.history = []

    def fit(self, interactions, user_features, item_features, sample_weight,
            epochs, num_threads, verbose, validation_data, evaluate_every):
        rng = np.random.default_rng(self.random_state)
        train_loss = 1.0
        for epoch in range(1, epochs + 1):
            # fake loss that trends down with a bit of noise, just for illustration
            train_loss = max(0.01, train_loss - rng.uniform(0.02, 0.08))
            val_loss = train_loss + rng.uniform(0.0, 0.05)
            self.history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
            if verbose and epoch % max(1, epochs // 5) == 0:
                print(f"  epoch {epoch:>3}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")
        return self

    def generate_training_report(self, output_dir, experiment_name):
        report_path = Path(output_dir) / f"{experiment_name.replace(' ', '_')}_report.csv"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(self.history).to_csv(report_path, index=False)
        return str(report_path)

    def save_model(self, path):
        np.savez(path, no_components=self.no_components, final_loss=self.history[-1]["train_loss"])


def train_model_demo(train_matrices, test_matrices, config, epochs, exname, output_dir):
    """Same call pattern as the real train_model(), using MockACBModel."""
    print(f"Starting training: epochs={epochs} | threads={config['num_threads']}")
    model = MockACBModel(
        no_components     = config["no_components"],
        loss              = config["loss"],
        learning_rate     = config["learning_rate"],
        item_alpha        = 0.01,
        user_alpha        = 0.01,
        learning_schedule = config["learning_schedule"],
        random_state      = 4,
    )
    model.fit(
        interactions=train_matrices["interactions"],
        user_features=train_matrices["user_features"],
        item_features=train_matrices["item_features"],
        sample_weight=train_matrices["weight"],
        epochs=epochs,
        num_threads=config["num_threads"],
        verbose=config["verbosity"],
        validation_data=test_matrices["interactions"],
        evaluate_every=1,
    )
    report_path = model.generate_training_report(output_dir=output_dir, experiment_name=exname)
    print(f"Training completed. Report: {report_path}")
    return model, report_path


def save_model_demo(model, output_dir: Path) -> Path:
    """Same call pattern as the real save_model()."""
    dates = f'{datetime.now():%Y%m%d}'
    model_path = Path(output_dir) / "ACBmodel" / f'{dates}_models.npz'
    model_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_path))
    print(f"Model saved: {model_path}")
    return model_path


### 4.7 Run the demo pipeline end-to-end


In [ ]:
OUTPUT_DIR = Path("./demo_artifacts")

config_demo = {
    "no_components"    : 10,
    "loss"             : "warp",
    "learning_rate"    : 0.05,
    "epochs"           : 15,
    "num_threads"      : 4,
    "dtype"            : "float32",
    "learning_schedule": "adagrad",
    "verbosity"        : True,
}

start = time.perf_counter()
model, report_path = train_model_demo(
    train_matrices, test_matrices, config_demo,
    epochs=config_demo["epochs"], exname="Demo Training Run", output_dir=OUTPUT_DIR,
)
model_path = save_model_demo(model, OUTPUT_DIR)
elapsed = time.perf_counter() - start

n_users = data_merge[collect['user_col']].nunique()
n_items = data_merge[collect['item_col']].nunique()
n_interactions = len(data_merge)
sparsity = 1.0 - (n_interactions / (n_users * n_items))

summary = {
    "status": "SUCCESS",
    "training_time": elapsed,
    "model_path": str(model_path),
    "report_path": report_path,
    "data_stats": {
        "n_users": n_users,
        "n_items": n_items,
        "n_interactions": n_interactions,
        "sparsity": sparsity,
    },
}

print("\n" + "=" * 50)
print("TRAINING PIPELINE EXECUTION SUMMARY")
print("=" * 50)
for k, v in summary.items():
    print(f"{k:>15}: {v}")


### 4.8 Inspect the training curve

Since `generate_training_report` wrote a CSV, we can load and plot it directly — this mirrors how you'd inspect a real training report from `AryColBringModelTrainer`.


In [ ]:
report_df = pd.read_csv(report_path)
report_df.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", figsize=(7, 4),
                title="Mock training curve (illustrative only)")
report_df


## 5. Putting it together: `__call__` as a single orchestrating function

In the original class, calling `fx(epochs=..., exname=...)` triggers `__call__`, which runs every stage above in order and returns a summary dict. Below is that same orchestration written as a plain function that calls the Section 3 functions (real logic) — this is what you'd use once the real `cooprecsys` imports are available.


```python
def run_full_pipeline(data, test_ratio, user_feats, item_feats, output_dir,
                       epochs, exname,
                       GenQuasi_Lazy, DetectReco_Identifier, DMD, ACBmodel, logger) -> dict:
    """
    Runs the entire AryColBring training pipeline end-to-end:
    rate generation -> column detection/merge -> train/test split ->
    train -> save -> summary.

    This mirrors AryColBring_Train_Test.__call__() from the original script.
    """
    start_time = time.perf_counter()

    data_rate = generate_quasi_ratings(data, GenQuasi_Lazy, logger)
    collect, data_merge, item_feats = detect_and_merge(
        data, data_rate, item_feats, DetectReco_Identifier
    )
    train_matrices, test_matrices = split_train_test(
        data_merge, test_ratio, collect, user_feats, item_feats, DMD
    )
    model, report_path = train_model(
        train_matrices, test_matrices, load_config(_cfg), epochs, exname, output_dir, ACBmodel, logger
    )
    model_path = save_model(model, output_dir, logger)

    elapsed = time.perf_counter() - start_time
    n_users = data_merge[collect['user_col']].nunique() if not data_merge.empty else 0
    n_items = data_merge[collect['item_col']].nunique() if not data_merge.empty else 0
    n_interactions = len(data_merge)
    sparsity = 1.0 - (n_interactions / (n_users * n_items)) if (n_users * n_items) > 0 else 0.0

    return {
        "status": "SUCCESS",
        "training_time": elapsed,
        "model_path": str(model_path),
        "report_path": report_path,
        "data_stats": {
            "n_users": n_users,
            "n_items": n_items,
            "n_interactions": n_interactions,
            "sparsity": sparsity,
        },
    }
```

Notice this function does nothing new — it just **calls the Section 3 functions in the right order** and assembles their outputs into one summary dict. That's the entire value the original `__call__` method adds: sequencing, not new logic. This is exactly what Section 4's demo does too (just spelled out cell-by-cell instead of hidden inside one function, so you could watch each stage's output).


## 6. The command-line interface (`main()`)

The bottom of the original script lets you run the whole pipeline from a terminal:

```bash
python t01_advisor.py -d ./data/sampledata.parquet -t 0.25 -e 50 -n "Test experiment"
```

Here's what each flag maps to:

| Flag | Meaning | Maps to |
|---|---|---|
| `-d / --datapath` | Path to the interaction data file | `AryColBring_Train_Test.Data` |
| `-t / --testratio` | Fraction of rows held out for testing (e.g. `0.2`) | `testratio` |
| `-u / --userfeature` | Comma-separated list of user feature columns | `UserFeats` |
| `-i / --itemfeature` | Comma-separated list of item feature columns | `ItemFeats` |
| `-o / --outputdir` | Where to save the trained model & report | `output_dir` |
| `-e / --epochs` | Number of training epochs | `epochs` argument to `train()` |
| `-n / --experimentname` | Label used in the saved report filename | `exname` argument to `train()` |

```python
def main() -> None:
    splitter = lambda s: [item.strip() for item in s.split(',')]
    parser = ArgumentParser(description="Train AryColBring model")
    parser.add_argument("-d", "--datapath", type=str, required=False, default=None,
                         help="Path to training data")
    parser.add_argument("-t", "--testratio", type=float, required=True,
                         help="Test-set ratio (e.g. 0.2)")
    parser.add_argument("-u", "--userfeature", type=splitter, default=None,
                         help="Comma-separated list of user feature columns")
    parser.add_argument("-i", "--itemfeature", type=splitter, default=None,
                         help="Comma-separated list of item feature columns")
    parser.add_argument("-o", "--outputdir", type=str, default="artifacts",
                         help="Directory to save artifacts")
    parser.add_argument("-e", "--epochs", type=int, required=False,
                         help="Number of training epochs")
    parser.add_argument("-n", "--experimentname", type=str, default="ACB Training Run",
                         help="Name of the current experiment run")
    args = parser.parse_args()

    datapath = Path(args.datapath) if args.datapath else (LocDir.parent / 'data' / 'sampledata.parquet')

    fx = AryColBring_Train_Test(UserFeats=args.userfeature, ItemFeats=args.itemfeature,
                                 output_dir=args.outputdir)
    fx.Data = datapath
    fx.testratio = args.testratio
    results = fx(epochs=args.epochs, exname=args.experimentname)
    # ... logging of the results summary ...
```

**Beginner note:** `splitter = lambda s: [item.strip() for item in s.split(',')]` is used as the `type=` for `argparse` — it lets you pass `-u "EmployeeAge, EmployeeGender"` on the command line and receive a clean Python list `["EmployeeAge", "EmployeeGender"]` inside the script.


## 7. Function-based vs. class-based: why did the original use a class?

Rewriting everything as standalone functions (Sections 3–5) makes the *logic* easier to follow line-by-line, but it also loses a few things the class was doing for you:

| What the class gives you | What you lose if you go pure-function |
|---|---|
| **State management** — `self.Data`, `self._TRAIN`, `self._TEST`, `self.ACBmodel` all live together on one object | You must manually thread every intermediate result through function arguments and return values (as Section 4 does) |
| **Validation on assignment** — `Data` and `testratio` are `@property` setters, so `fx.Data = "bad/path"` raises immediately | A function only validates when it's actually called; nothing stops you from skipping a validation function |
| **A clean external API** — a user of the class only needs to know `Data`, `testratio`, and `fx(epochs=..., exname=...)` | A user of the functional version needs to know the correct call *order* for ~7 functions and pass many shared variables (`collect`, `user_feats`, `item_feats`, ...) between them |
| **Encapsulation** — internal helper methods (`_RateProgress`, `_brokedata`, ...) are clearly marked private by the leading underscore | All functions are equally "public" unless you add your own naming convention |

**Rule of thumb:** if a bundle of steps always run together, share a lot of state, and are used by other code as one unit, a class is often the right tool. If you're just trying to *understand or teach* the steps, breaking it into functions (like this notebook) is the more readable form. Good production code often keeps the class as the public interface, while implementing each internal method as a thin wrapper around a well-tested standalone function — giving you both benefits at once.


## 8. Appendix — original class, unmodified

For side-by-side comparison, here is the original `AryColBring_Train_Test` class exactly as it appears in `t01_advisor.py` (shown as text, not executed, since it depends on the private `cooprecsys` package).


In [ ]:
ORIGINAL_CLASS_SOURCE = """
class AryColBring_Train_Test:
    # High-level training pipeline wrapper for AryColBring model.
    # Handles data loading, training, evaluation, and reporting.
    def __init__(self, UserFeats=None, ItemFeats=None, testratio=float(), output_dir=None):
        if output_dir is None:
            self.output_dir = LocDir.parent / _cfg.get('PATHS', 'output_dir')
        else:
            self.output_dir = Path(output_dir)
        self.config = self._load_config()
        self._Data = pd.DataFrame([])
        self.DataMerge = pd.DataFrame([])
        self.data_rate = pd.DataFrame([])
        self.UserFeats = UserFeats or ['EmployeeAge', 'EmployeeGender', 'Resistant', 'IsAllergic', 'VitalityDays']
        self.ItemFeats = ItemFeats or ['ProductPrice', 'Quantity', 'Discount', 'TotalPrice', 'Class']
        self._testratio = testratio
        self.Collect = dict()
        self._TRAIN = None
        self._TEST = None
        self.output_dir.mkdir(parents=True, exist_ok=True)

    # ... property setters for Data / testratio ...
    # ... _load_config, _RateProgress, _RatePosthoc, _ttprocess, _brokedata ...
    # ... train, save, __call__ ...

    # See the full source at:
    # https://github.com/masterofray/cooprecsys/blob/dev/test/arycolbring_tests/t01_advisor.py
"""
print(ORIGINAL_CLASS_SOURCE)


## Summary

- The pipeline has 3 real "thinking" stages (rating generation, column detection/merge, matrix conversion) and 2 mechanical stages (split, train/save) — everything else is orchestration.
- Sections 3–5 preserve the **exact real logic**, just reshaped into functions.
- Section 4 is a **safe playground**: swap `mock_*` functions for the real `cooprecsys` imports once you have the package installed and your real data path, and the same function calls will work against real data.
- Section 7 explains why the *production* version reasonably stays a class, even though functions are the friendlier teaching format.

**Next step to run this for real:**
```bash
pip install git+https://github.com/masterofray/cooprecsys.git@dev
python t01_advisor.py -d ./data/sampledata.parquet -t 0.25 -e 50 -n "Test experiment"
```
